In [1]:
#Querying notebook
#While the ingestion is running (or after it finishes), create another notebook (see persinsent_rag.ipynb for reference).

#Connect to the same database:
    
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [12]:
#Check how many documents are in the index:
sqlite_index.count()

85

In [11]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
 'I missed the first homework - can I still get a certificate?',
 'Homework: Why does the content keep changing?',
 'What happens to code saved in Codespaces if I do not commit it?']

In [10]:
#Because our RAG is modular, we just swap the search index - the rest of the code stays the same:

from dotenv import load_dotenv
from google import genai  # Cambiamos openai por la librería de Google
from rag_helper import RAGBase

# 1. Cargamos las variables de entorno primero (.env)
load_dotenv()

# 2. Inicializamos el cliente oficial de Gemini
gemini_client = genai.Client()

# 3. Instanciamos el asistente con tu índice de SQLite y el cliente de Google
assistant = RAGBase(
    index=sqlite_index,  # Tu nuevo índice persistente de SQLite
    llm_client=gemini_client,  # El cliente correcto que configuramos en la clase
)


In [7]:
#This code skips both the fit call and the data loading. The index is already populated by the ingestion notebook, 
#so we just connect to the database file.

answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can still join the course after it started. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [8]:
sqlite_index.close()